In [1]:
import os
import sys

from osgeo import gdal

import cv2
import numpy as np 
import skimage.filters
import skimage.color
import skimage.io
import matplotlib.pyplot as plt
import shapely
from shapely.geometry.polygon import LinearRing, Polygon, LineString, Point
from shapely.geometry import MultiPolygon
import pandas as pd
from area import area
from shapely.ops import cascaded_union
from PIL import Image
from keras.layers import *
from pyproj import Transformer
#cwd = os.getcwd()
#os.environ["PATH"] += os.pathsep + "C:/LAB/supporting/DOT"
#os.environ["PATH"] += os.pathsep + cwd + "\\DOT"

In [ ]:
#imagelst=os.listdir('C:/IMAGE/Foliage/city_Bangalore/Bangalore/')
path_input='C:/IMAGE/Foliage/test/test/'
path_output = 'C:/IMAGE/Foliage/test/test/SUBGRID_DATA_IMAGEBASED_TEST.txt'

path_model = 'C:/IMAGE/Foliage/Transfer Learning_22-01-2024/building-detector-v11-colored_transfer_S4.h5'

In [2]:
def point_in_poly(poly, point_lat, point_lon):

    point_chk = Point(point_lon, point_lat)
    if(point_chk.within(poly)):
        return 1
    else:
        return 0
    
def polyToPointList(poly):
    pnt_list = []
    x, y = poly.exterior.xy
    for i, x_ in enumerate(x):
        pnt_list.append([x[i], y[i]])
    return pnt_list


def lineToPointList(line_):
    pnt_list = []
    if(line_.geom_type == 'MultiLineString'):
        for it in line_:
            x, y = it.xy
            for i, x_ in enumerate(x):
                pnt_list.append([x[i], y[i]])
    else:
        x, y = line_.xy
        for i, x_ in enumerate(x):
            pnt_list.append([x[i], y[i]])
    return pnt_list


def getRouteLen(ptlist):
    x, y = [i[0] for i in ptlist], [i[1] for i in ptlist]
    len_ = 0
    j = 0
    for i, x_ in enumerate(x):
        if(i <= len(x) - 2):
            len_ = len_ + \
                great_circle((y[i], x[i]), (y[i + 1], x[i + 1])).meters

    return round(len_, 3)

TRAN_3857_TO_4326 = Transformer.from_crs( "EPSG:3857","EPSG:4326")

def mytransform(lon, lat):
    
    return TRAN_3857_TO_4326.transform(lon, lat)

In [3]:
from tensorflow.keras.models import load_model
#model_loaded1 = load_model('C:/IMAGE/Foliage/modeldata/building-detector-v11-delhi_101.h5')
model_loaded1 = load_model(path_model)

In [4]:
def ClassifyArea_Parallel(image_path):

    pred_lst=[]
    val_lst=[]
    TotalCount=0
    BuidingCount=0
    
    width=500
    height=500
    chunk_size=50
    y_size=chunk_size 
    x_size=chunk_size 

    img_size = 224
    img = cv2.imread(image_path)
    imS = cv2.resize(img, (500,500)) 

    #####################
    imagelist=[]
    all_grids=[]
    building_grids=[]
    other_grids=[]
    idx=0
    for ys in np.arange(0,height,y_size):
        for xs in np.arange(0,width,x_size):
            if(ys<=height-y_size and xs<=width-x_size):
                #print(ys,xs)
                idx=idx+1
                start_point = (ys,xs)
                end_point = (ys+y_size,xs+x_size)

                all_grids.append((xs,ys))

                crop_img = imS[ys:ys+y_size, xs:xs+x_size][...,::-1] 
                resized_arr = cv2.resize(crop_img, (img_size, img_size))

                imagelist.append(resized_arr)

    img_size = 224
    imagelist_in = np.array(imagelist) / 255
    imagelist_in.reshape(-1, img_size, img_size, 1)

    predict_x = model_loaded1.predict(imagelist_in,verbose = 0)

    classes_x=np.argmax(predict_x,axis=1)

    #plt.show()
    return classes_x,predict_x

In [5]:
from datetime import datetime


imagelst=os.listdir(path_input)


st=datetime.now()
print("Started : {}".format(str(st)))

sindex=0
inpaths=[]


GRID_ID=[]
FPATH=[]

RESP_GRID_ID=[]
RESP_PRED=[]
RESP_VAL=[]


grid_idlst=[]
point_lat=[]
point_lon=[]
point_image=[]
point_build_prob=[]
CENTER_LAT=[]
CENTER_LON=[]

for idx,IMAGE_NAME in enumerate(imagelst):

    if(IMAGE_NAME.endswith(".tif")):
        
        #FPATH_VAL='C:/IMAGE/Foliage/city_Bangalore/Bangalore/'+IMAGE_NAME
        FPATH_VAL=path+IMAGE_NAME
        FPATH.append(FPATH_VAL)

        classes_x,predict_x=ClassifyArea_Parallel(FPATH_VAL)
        RESP_PRED.append(classes_x)
        RESP_VAL.append(predict_x)


        preds=[x[0] for x in predict_x]

        #####################################################
        # CREATING POINT WISE DATA

        ds = gdal.Open(FPATH_VAL)
        width = ds.RasterXSize
        height = ds.RasterYSize
        gt = ds.GetGeoTransform()

        minx = gt[0]
        miny = gt[3] + width*gt[4] + height*gt[5] 
        maxx = gt[0] + width*gt[1] + height*gt[2]
        maxy = gt[3]

        #print(width, height)
        #print(gt)
        #print("MinX, MinY: ", minx, miny)
        #print("MaxX, MaxY: ", maxx, maxy)

        image_path=FPATH_VAL
        width=500
        height=500
        chunk_size=50
        y_size=chunk_size 
        x_size=chunk_size 

        img_size = 224
        img = cv2.imread(image_path)
        imS = cv2.resize(img, (500,500))     

        ind=0

        for ys in np.arange(0,height,y_size):
            for xs in np.arange(0,width,x_size):
                if(ys<=height-y_size and xs<=width-x_size):
                    #print(ys,xs)

                    start_point = (ys,xs)
                    end_point = (ys+y_size,xs+x_size)

                    #print(start_point)
                    #print(miny+ys+25,minx+xs+25)
                    res=mytransform(minx+xs+25, maxy-ys-25)
                    #res=mytransform(minx+xs+25, miny+ys+25)
                    point_lat.append(res[0])
                    point_lon.append(res[1])

                    point_image.append(IMAGE_NAME)
                    point_build_prob.append(preds[ind])     

                    #print("For grid image ", ind, " of Mumbai image ", idx, ".tif")
                    #print("EPSG 3857: ", [minx+xs+25, maxy-ys-25], " EPSG 4326: ", res)
                    
                    ind=ind+1

        #####################################################

        print(str(idx),end="|")



et=datetime.now()
print("\nEnded : {}".format(str(et)))

print('\n',str(et-st))

Started : 2024-08-19 12:01:23.187246
0|1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|19|20|21|22|23|24|25|26|27|28|29|
Ended : 2024-08-19 12:01:37.066054

 0:00:13.878808


In [6]:
data=pd.DataFrame()

data["IMAGE_NAME"]=point_image
data["PT_LAT"]=point_lat
data["PT_LON"]=point_lon
data["BUILD_PROB"]=point_build_prob

In [ ]:
#print(point_lat)
#print(point_lon)

In [7]:
data.to_csv(path_output, sep='\t')

In [ ]:
'''
def getAvg_lat(df,imname):
    C2_LAT=df.loc[df['IMAGE_NAME']==imname]['PT_LAT'].mean()
    return C2_LAT
def getAvg_lon(df,imname):
    C2_LON=df.loc[df['IMAGE_NAME']==imname]['PT_LON'].mean()
    return C2_LON
'''

In [ ]:
#data=pd.read_csv('E:/IMAGEPRO/BASELINE_DELHI_UC/SUBGRID_DATA_IMAGEBASED.txt',sep='\t')

In [ ]:
#datagrp=data.loc[data['BUILD_PROB']>=0.25].groupby(['IMAGE_NAME']).agg({'PT_LAT':'mean','PT_LON':'mean'}).reset_index()

In [ ]:
#datagrp.columns=['IMAGE_NAME','C2_LAT','C2_LON']

In [ ]:
#datagrp.head(10)

In [ ]:
#data=pd.merge(data,datagrp,how='left',on='IMAGE_NAME')

In [ ]:
#data.to_csv('E:/IMAGEPRO/BASELINE_DELHI_UC/SUBGRID_DATA_IMAGEBASED_C2_25per.txt',sep='\t')